In [ ]:
!pip install unsloth

# Upgrade Unsloth from the latest repository
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"


  Using cached unsloth-2025.2.15-py3-none-any.whl.metadata (57 kB)
Using cached unsloth-2025.2.15-py3-none-any.whl (188 kB)
Found existing installation: unsloth 2025.2.15
Uninstalling unsloth-2025.2.15:
  Successfully uninstalled unsloth-2025.2.15
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-3q2z5uvz/unsloth_47a87cb5a39646adadb43b9642c65ce5
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-3q2z5uvz/unsloth_47a87cb5a39646adadb43b9642c65ce5
  Resolved https://github.com/unslothai/unsloth.git to commit 14c9be1d7160162a90ce7a9a6cae36965563a0e6
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for unsloth: filename=unsloth-2025.2.15-py3-none-any.whl size=188380 sha256=2529aaff6e5a9128aa3bca51498eaed08f62c8971d0a787e4b2f35ea54a7c08f
  Stored in directory: /tmp/pip-ephem-wheel-cache-b0dy10im/wheels/d1/17/0

In [ ]:
#!pip uninstall bitsandbytes -y
!pip install --upgrade bitsandbytes


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "julpwls/llama-3.2-course-recommender"  # Twoja nazwa repo

# Pobranie modelu i tokenizera z Hugging Face Hub
tokenizer = AutoTokenizer.from_pretrained(model_name, use_auth_token=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="cuda",  # Wymuszenie użycia GPU
    torch_dtype=torch.float16,  # Wymuszenie float16
    use_auth_token=True
)

print(" Model załadowany i gotowy do użycia!")


/usr/local/lib/python3.11/dist-packages/transformers/models/auto/tokenization_auto.py:823: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/models/auto/auto_factory.py:471: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
Unused kwargs: ['_load_in_4bit', '_load_in_8bit', 'quant_method']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/97.3M [00:00<?, ?B/s]

 Model załadowany i gotowy do użycia!


In [ ]:
import os
from huggingface_hub import login

hf_token = os.getenv("HF_TOKEN")

login(token=hf_token, add_to_git_credential=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import json
from collections import defaultdict
from datasets import Dataset
from transformers import AutoTokenizer

json_path = "/content/drive/My Drive/half_from_half.json"  # Zmień ścieżkę jeśli plik jest w podfolderze

with open(json_path, 'r', encoding='utf-8') as file:
    raw_data = json.load(file)

#  Deduplicate courses & merge similar ones
course_dict = defaultdict(lambda: {"course_goals": set(), "course_results": set(), "course_program": set()})

for course in raw_data:
    course_name = course["course_name"]
    course_dict[course_name]["course_goals"].update(course["course_goals"].split(". "))
    course_dict[course_name]["course_results"].update(course["course_results"].split(". "))
    course_dict[course_name]["course_program"].update(course["course_program"].split(". "))

#  Convert sets back to formatted strings
for course_name, details in course_dict.items():
    details["course_goals"] = "\n- " + "\n- ".join(sorted(details["course_goals"]))
    details["course_results"] = "\n- " + "\n- ".join(sorted(details["course_results"]))
    details["course_program"] = "\n- " + "\n- ".join(sorted(details["course_program"]))

#  Load tokenizer
model_id = "meta-llama/Llama-3.2-3B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
EOS_TOKEN = tokenizer.eos_token  # End of sequence token


def format_training_data():
    structured_data = []
    for course_name, details in course_dict.items():
        prompt = f"""### Instrukcja:
Jesteś asystentem AI przeszkolonym do rekomendowania kursów edukacyjnych na podstawie poziomu studiów użytkownika oraz jego obszaru zainteresowań. Rekomenduj najlepszy kurs dla użytkownika, zapewniając, że:
- Kurs odpowiada poziomowi studiów użytkownika (Studia licencjackie/inżynierskie, Studia magisterskie, Jednolite studia magisterskie, Studia doktoranckie, Studia podyplomowe)
- Oferuje ustrukturyzowaną ścieżkę nauki
- Wyjaśnia kluczowe zagadnienia i oczekiwane rezultaty
- Nie powtarza danych wejściowych użytkownika


### Wejście:
Poziom studiów: {{study_level}}
Obszar studiów: {course_name}

### Odpowiedź:
Najlepszy kurs dla studenta na poziomie {{study_level}}, który studiuje w obszarze {course_name}, to:
"{course_name}"


### Cele kursu:
{details['course_goals']}

#### Oczekiwane rezultaty:
{details['course_results']}

#### Zakres tematów:
{details['course_program']}

"""
        structured_data.append({"text": prompt + EOS_TOKEN})

    return structured_data

#  Convert dataset to Hugging Face format
dataset = Dataset.from_list(format_training_data())


In [ ]:
dataset[0]

In [ ]:
import torch
from unsloth import FastLanguageModel, is_bfloat16_supported
from peft import LoraConfig
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset


# Set parameters
max_seq_length = 2048
model_name = "meta-llama/Llama-3.2-3B"

# Load model with Unsloth (4-bit quantization)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    dtype=None,
)

# Apply PEFT (LoRA) for memory-efficient tuning
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "up_proj", "down_proj", "o_proj", "gate_proj"],
    use_rslora=True,
    use_gradient_checkpointing="unsloth",
    random_state=42,
    loftq_config={"quant_type": "fp4"}
)

# Print trainable parameters
print(model.print_trainable_parameters())

# Training arguments
training_args = TrainingArguments(
    learning_rate=5e-4,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=1,
    optim="adamw_torch_fused",
    weight_decay=0.01,
    warmup_steps=20,
    output_dir="llama-3.2-course-recommender",
    push_to_hub=True,
    seed=0,
    report_to="none",
)

# Initialize trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=1,
    packing=True,
    args=training_args,
)

ModuleNotFoundError: No module named 'unsloth'

In [ ]:
# Start training
trainer.train()


### Model Inference

In [ ]:
import torch
from unsloth import FastLanguageModel
from transformers import pipeline, AutoTokenizer

model = FastLanguageModel.for_inference(model).to("cuda")

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.float16,  # Use float16 for low VRAM usage
    do_sample=True,  # Umożliwia bardziej różnorodne odpowiedzi
    temperature=0.7,  # Balans między kreatywnością a sensownością
    top_p=0.9,  # Ograniczenie generowania do najbardziej prawdopodobnych słów
)

print(" Model loaded successfully on GPU for inference!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


Device set to use cuda


 Model loaded successfully on GPU for inference!


In [ ]:

def recommend_course(study_level, topic):
    prompt = f"""### Instrukcja:
Zarekomenduj kurs na poziomie {study_level} który interesuje się {topic}.

### Odpowiedź:
"""
    output = pipe(prompt, temperature=0.7, top_k=50)
    return output[0]['generated_text']

#Test the model
print(recommend_course("Studia inżynierskie", "autocad"))

